# 05 — Difference in Differences

Synthetic DiD notebook with parallel trend diagnostics and pitfalls.

**Project:** Marketing Analytics Causal & LTV Lab  
**Style:** Hands-on, advanced, interview-ready notebook  
**How to use:** Run cell by cell, inspect outputs, then discuss interpretation and pitfalls.


## Main notebook code

Run this notebook and then we will discuss the output, assumptions and pitfalls.


In [ ]:
import warnings; warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.formula.api as smf
RANDOM_STATE=42; rng=np.random.default_rng(RANDOM_STATE); SYNTHETIC_DIR=Path('../data/synthetic'); SYNTHETIC_DIR.mkdir(parents=True, exist_ok=True)
weeks=pd.date_range('2024-01-01',periods=80,freq='W'); regions=[f'region_{i}' for i in range(20)]; treated=set(regions[:8]); intervention=weeks[45]
rows=[]
for r in regions:
    fe=rng.normal(0,10); tr=r in treated
    for t,w in enumerate(weeks):
        rev=100+fe+0.5*t+8*np.sin(2*np.pi*t/52)+(12 if tr and w>=intervention else 0)+rng.normal(0,8)
        rows.append({'week':w,'region':r,'treated_region':int(tr),'post':int(w>=intervention),'revenue':rev})
did=pd.DataFrame(rows); did['treated_x_post']=did.treated_region*did.post
trend=did.groupby(['week','treated_region']).revenue.mean().reset_index()
plt.figure(figsize=(10,4))
for g,tmp in trend.groupby('treated_region'): plt.plot(tmp.week,tmp.revenue,label='treated' if g else 'control')
plt.axvline(intervention,linestyle='--'); plt.legend(); plt.title('Treated vs Control'); plt.show()
model=smf.ols('revenue ~ treated_region + post + treated_x_post',data=did).fit(cov_type='HC3'); print(model.summary()); print('DiD:', model.params['treated_x_post'])
twfe=smf.ols('revenue ~ treated_x_post + C(region) + C(week)',data=did).fit(cov_type='HC3'); print('TWFE:', twfe.params['treated_x_post'])
pre=did[did.post==0].copy(); pre['time_index']=pre.groupby('region').cumcount(); print(smf.ols('revenue ~ treated_region * time_index',data=pre).fit(cov_type='HC3').summary().tables[1])
did.to_csv(SYNTHETIC_DIR/'synthetic_did_regions.csv',index=False)


## Discussion prompts

1. What assumption is strongest here?
2. Which pitfall would break the conclusion?
3. How would you explain this to a non-technical stakeholder?
